In [0]:
import json
from pyspark.sql.functions import col, current_timestamp, from_json, udf
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType

dbutils.widgets.text("catalog", "dbr_dev")
dbutils.widgets.text("schema", "janvander0912_bronze")
dbutils.widgets.text("storage_account", "dlspl21databricks")
dbutils.widgets.text("container", "janvander0912")
dbutils.widgets.text("volume", "raw_data")

dbutils.widgets.text("secret_scope", "default2")           
dbutils.widgets.text("secret_key", "janvander0912-evh-cs") 
dbutils.widgets.text("evh_name", "janvander0912-evh")      

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
storage_account = dbutils.widgets.get("storage_account")
container = dbutils.widgets.get("container")
volume = dbutils.widgets.get("volume")
secret_scope = dbutils.widgets.get("secret_scope")
secret_key = dbutils.widgets.get("secret_key")
evh_name = dbutils.widgets.get("evh_name")

In [0]:
base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/{volume}/"
checkpoint_path = f"{base_path}_checkpoints/wiki_events_v3/"
table_name = f"{catalog}.{schema}.wikipedia_edits_bronze"

print(f"Retrieving the secret and configuring the connection for the table: {table_name}...")
conn_string = dbutils.secrets.get(scope=secret_scope, key=secret_key)

endpoint_raw = conn_string.split("Endpoint=sb://")[1].split("/")[0].split(";")[0]
bootstrap_server = f"{endpoint_raw}:9093"

print(f" -> Kafka server detected: {bootstrap_server}")

In [0]:
kafka_options = {
    "kafka.bootstrap.servers": bootstrap_server,
    "subscribe": evh_name,
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{conn_string}";',
    "startingOffsets": "earliest"
}

df_raw = spark.readStream.format("kafka").options(**kafka_options).load()

event_schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("user", StringType(), True),
    StructField("article_title", StringType(), True),
    StructField("wiki_domain", StringType(), True),
    StructField("is_bot", BooleanType(), True),
    StructField("length_change", IntegerType(), True),
    StructField("event_timestamp", StringType(), True)
])

In [0]:
@udf(returnType=StringType())
def calculate_edit_impact_udf(length_change, is_bot):
    if is_bot:
        return "AUTOMATED_BOT_MAINTENANCE"
    elif length_change > 500:
        return "MAJOR_CONTENT_ADDITION"
    elif length_change < -500:
        return "POTENTIAL_VANDALISM_OR_SHRINK"
    else:
        return "MINOR_HUMAN_EDIT"


df_processed = (df_raw
    .selectExpr("CAST(value AS STRING) as json_payload")
    .select(from_json("json_payload", event_schema).alias("data"))
    .select("data.*")
    # Calling our legitimate UDF function
    .withColumn("edit_impact_category", calculate_edit_impact_udf(col("length_change"), col("is_bot")))
    .withColumn("ingestion_timestamp", current_timestamp())
)

# DELTA ENROLLMENT
print("Starting a stream that writes to the Delta table...")
query_wiki = (df_processed.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True) 
    .toTable(table_name)
)

query_wiki.awaitTermination()
print("Success! All events from Event Hub have been safely saved to the Delta table.")


# Verification
# print("\nPreview of saved data from Wikipedia")
# display(spark.sql(f"""
#     SELECT event_id, user, article_title, length_change, is_bot, edit_impact_category, ingestion_timestamp 
#     FROM {table_name} 
#     ORDER BY event_id DESC 
#     LIMIT 15
# """))